# 🗣️ Notebook 1: What is a Gossip Protocol?

**The big question:** *"In a cluster of 1000 nodes, how do you tell everyone something — without sending 1000 messages from one place?"*

A naïve answer: have one "leader" broadcast to all 999 others. That works for 10 nodes. At 10,000 nodes it falls over: the leader is a bottleneck and a single point of failure.

A **gossip protocol** copies what humans do at a party: every few seconds, each node picks a small random set of peers and tells them what it knows. Those peers do the same. Information spreads exponentially — like a rumour.

In this notebook we compare:

1. 🟥 **BAD** — central broadcaster, one node tells everyone.
2. 🟩 **GOSSIP** — each node tells `fanout=3` random peers per round.

We measure how many rounds it takes to reach every node.

## Learning objectives
- Understand the epidemic / push-gossip algorithm.
- Observe O(log N) rounds to converge regardless of cluster size.
- Visualize convergence.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/gossip-protocol
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook).

In [ ]:
import random
random.seed(0)

N = 100             # nodes in the cluster
FANOUT = 3          # peers each node tells per round

## 🟥 Approach 1: Central broadcaster

One node tries to broadcast to all the others. To make it interesting we say it can only send 3 messages per round (limited bandwidth).

In [ ]:
def central_broadcast(n, msgs_per_round=3):
    informed = {0}      # node 0 starts with the news
    rounds = 0
    while len(informed) < n:
        rounds += 1
        # node 0 picks 3 not-yet-informed nodes per round
        candidates = [i for i in range(n) if i not in informed]
        for target in candidates[:msgs_per_round]:
            informed.add(target)
    return rounds

print(f"central broadcast converged in {central_broadcast(N)} rounds")

## 🟩 Approach 2: Gossip (push)

Each round, **every informed node** picks `FANOUT` random peers and shares the news. Information doubles (well, multiplies by ~`fanout`) every round.

In [ ]:
def gossip(n, fanout=3):
    informed = {0}
    history = [len(informed)]
    rounds = 0
    while len(informed) < n:
        rounds += 1
        new_informed = set(informed)
        for src in informed:
            for _ in range(fanout):
                peer = random.randrange(n)
                new_informed.add(peer)
        informed = new_informed
        history.append(len(informed))
    return rounds, history

rounds, history = gossip(N, FANOUT)
print(f"gossip converged in {rounds} rounds with fanout={FANOUT}")
print("informed per round:", history)

## 📈 Convergence shape: S-curve

The number of informed nodes follows a classic *epidemic* S-curve: slow start (only one node knows), then explosive growth, then a long tail as the last few nodes hear the rumour.

In [ ]:
import matplotlib.pyplot as plt

# Run gossip for several cluster sizes to show O(log N) scaling.
sizes = [50, 100, 500, 1000, 5000]
for n in sizes:
    _, h = gossip(n, FANOUT)
    plt.plot(range(len(h)), [x / n for x in h], label=f"N={n}")
plt.xlabel("round")
plt.ylabel("fraction of nodes informed")
plt.title(f"Gossip convergence (fanout={FANOUT})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# How does fanout affect convergence?
for f in (1, 2, 3, 5, 10):
    rounds, _ = gossip(1000, f)
    print(f"fanout={f:2d}  ->  {rounds} rounds to reach 1000 nodes")

## ✅ Recap

- A **central broadcaster** scales linearly: `N` nodes → `N` messages from the leader.
- **Gossip** converges in roughly `O(log N)` rounds. Doubling cluster size adds only a few rounds.
- Bigger fanout → faster convergence, more bandwidth used per round.
- Real systems (Cassandra, Consul, Serf, DynamoDB) use gossip to spread membership and failure info because it has **no leader**, **no bottleneck**, and is robust to message loss.